# Backprojection Evaluation
Comparing GroundingDINO + backprojection estimated 3D positions against AI2-THOR ground truth metadata.

## Import

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
sys.path.append('..')

from perception.backprojection import get_ai2_thor_intrinsics, backproject_detections
from perception.detector import load_detector, detect_objects, boxes_to_pixel

# settings
DATA_DIR   = Path('../data/episodes/d1_exploration')
WIDTH, HEIGHT, FOV = 640, 480, 90
STEP = "step028"

/home/sungin1991/anaconda3/envs/embodied-ai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/sungin1991/anaconda3/envs/embodied-ai/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/sungin1991/anaconda3/envs/embodied-ai/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/sungin1991/anaconda3/envs/embodied-ai/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_

## 1. Ground Truth — Visible Objects
Load AI2-THOR metadata and extract position of all visible objects.

In [ ]:
with open(DATA_DIR / f'{STEP}_metadata.json') as f:
    metadata = json.load(f)

gt = {
    obj['objectType']: obj['position']
    for obj in metadata['objects']
    if obj['visible']
}
print(f"Visible objects in ground truth: {len(gt)}")
for name, pos in gt.items():
    print(f"  {name:<25} x={pos['x']:.3f} y={pos['y']:.3f} z={pos['z']:.3f}")

Visible objects in ground truth: 5
  Apple                     x=-0.465 y=1.151 z=0.476
  Book                      x=0.155 y=1.099 z=0.617
  CounterTop                x=-0.079 y=1.147 z=-0.001
  CreditCard                x=-0.464 y=1.100 z=0.865
  Floor                     x=0.000 y=0.000 z=0.000


## 2. Detection + Backprojection
Run GroundingDINO on the same image and backproject detected objects into 3D.

In [ ]:
depth_map = np.load(DATA_DIR / f'{STEP}_depth_raw.npy')

CONFIG_PATH = "../external/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
WEIGHTS_PATH = "../weights/groundingdino_swint_ogc.pth"

model = load_detector(config_path = CONFIG_PATH, weights_path=WEIGHTS_PATH)
image_source, boxes, logits, phrases = detect_objects(
    model, str(DATA_DIR / f'{STEP}_rgb.png')
)

h, w = image_source.shape[:2]
pixel_boxes = boxes_to_pixel(boxes, w, h)
K = get_ai2_thor_intrinsics(WIDTH, HEIGHT, FOV)
results = backproject_detections(pixel_boxes, depth_map, K, phrases)

print(f"Detected + backprojected: {len(results)} objects")
for r in results:
    print(f"  {r['object']:<25} X={r['X']:.3f} Y={r['Y']:.3f} Z={r['Z']:.3f}")

final text_encoder_type: bert-base-uncased


/home/sungin1991/anaconda3/envs/embodied-ai/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[INFO] Detector loaded on cuda


/home/sungin1991/anaconda3/envs/embodied-ai/lib/python3.10/site-packages/transformers/modeling_utils.py:942: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/home/sungin1991/anaconda3/envs/embodied-ai/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/sungin1991/anaconda3/envs/embodied-ai/lib/python3.10/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Detected + backprojected: 9 objects
  table                     X=0.455 Y=0.356 Z=0.754
  bowl plate                X=0.467 Y=0.347 Z=1.422
  pot chair                 X=0.732 Y=0.308 Z=0.771
  cup mug bottle            X=0.473 Y=0.353 Z=0.785
  chair                     X=3.218 Y=0.381 Z=3.479
  cup mug bottle bowl pan pot X=3.218 Y=0.655 Z=3.491
  plate pan pot drawer      X=0.596 Y=0.310 Z=1.256
  drawer                    X=-0.734 Y=0.118 Z=1.016
  table                     X=0.528 Y=0.355 Z=0.875


/home/sungin1991/anaconda3/envs/embodied-ai/lib/python3.10/site-packages/groundingdino/models/GroundingDINO/transformer.py:862: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


## 3. Match & Compute Error
Match detected object names to ground truth object types and compute depth estimation error.

In [ ]:
# 어느 스텝에서 HOME_OBJECTS랑 겹치는 visible 물체가 있는지 확인
home = {"cup","bowl","drawer","bottle","plate","mug","pan","pot","chair","table"}
for step in range(10):
    with open(DATA_DIR / f'step{step:03d}_metadata.json') as f:
        data = json.load(f)
    visible = [o['objectType'].lower() for o in data['objects'] if o['visible']]
    overlap = home & set(visible)
    if overlap:
        print(f"Step {step}: {overlap}")

In [ ]:
matches = []
for r in results:
    words = r['object'].lower().split()
    for gt_name, gt_pos in gt.items():
        gt_lower = gt_name.lower()
        for word in words:
            if len(word) > 2 and word in gt_lower:
                gt_z = abs(gt_pos['z'] - metadata['agent_position']['z'])
                error = abs(r['Z'] - gt_z)
                matches.append({
                    'detected': r['object'],
                    'gt_type':  gt_name,
                    'est_Z':    r['Z'],
                    'gt_Z':     gt_z,
                    'error':    error,
                })
                break

# deduplicate
seen = set()
unique_matches = []
for m in matches:
    if m['gt_type'] not in seen:
        seen.add(m['gt_type'])
        unique_matches.append(m)
matches = unique_matches

print(f"Matched {len(matches)} objects")
for m in matches:
    print(f"  {m['gt_type']:<20} est={m['est_Z']:.3f}m  gt={m['gt_Z']:.3f}m  error={m['error']:.3f}m")

Matched 0 objects


## 4. Visualization
Compare ground truth vs estimated depth (Z) per object, and plot absolute error.
Note: Z here refers to camera-frame depth, approximated from world coordinates.

In [ ]:
# visualize
if matches:
    labels   = [m['gt_type'] for m in matches]
    est_vals = [m['est_Z']   for m in matches]
    gt_vals  = [m['gt_Z']    for m in matches]
    errors   = [m['error']   for m in matches]

    x = np.arange(len(labels))
    width = 0.35

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # estimated vs ground truth
    ax1.bar(x - width/2, gt_vals,  width, label='Ground Truth', color='steelblue')
    ax1.bar(x + width/2, est_vals, width, label='Estimated',    color='coral')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, rotation=30, ha='right')
    ax1.set_ylabel('Depth Z (meters)')
    ax1.set_title('Ground Truth vs Backprojection Estimate')
    ax1.legend()

    # error per object
    ax2.bar(x, errors, color='salmon')
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels, rotation=30, ha='right')
    ax2.set_ylabel('Absolute Error (meters)')
    ax2.set_title('Estimation Error per Object')
    ax2.axhline(y=np.mean(errors), color='red', linestyle='--',
                label=f'Mean error: {np.mean(errors):.3f}m')
    ax2.legend()

    plt.tight_layout()
    plt.savefig('../data/episodes/d1_exploration/backprojection_eval.png', dpi=150)
    plt.show()
    print(f"\nMean error: {np.mean(errors):.3f}m")
else:
    print("No matches found between detected objects and ground truth.")

No matches found between detected objects and ground truth.


## 5. Interpretation
- Mean error shows overall backprojection accuracy
- Large errors likely due to GroundingDINO detecting wrong region (center pixel not on object)
- Sim-to-real gap: AI2-THOR renders perfect depth, real RGB-D sensors add noise